# 🧬 **Projeto SIEST: Sistema de Inteligência Epidemiológica e Socioespacial**
## Notebook 02: Geolocalização e Cruzamento Espacial (Camada Silver)

Este notebook é responsável por elevar os dados da Camada *Bronze* (dados brutos de morbidade) para a Camada *Silver* (dados refinados e georreferenciados).

**Estratégia Arquitetural e Resolução de Problemas:**
Durante a extração via FTP governamental, os servidores do DataSUS apresentaram bloqueios por limite de requisições (*Rate Limiting*) e ausência das coordenadas espaciais na base transacional de estabelecimentos.

Para contornar esta instabilidade e garantir a resiliência do *pipeline*, a equipe adotou a criação de um **Contrato de Dados Rígido (*Strict Schema*)** atrelado a uma fonte estática: o arquivo CSV do Cadastro Nacional de Estabelecimentos de Saúde (CNES), obtido via Portal de Dados Abertos. Esta abordagem permite o cruzamento determinístico das coordenadas geográficas (Latitude/Longitude) com os casos notificados.

In [13]:
import pandas as pd
import numpy as np
import os
import csv
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÕES GLOBAIS E DIRETÓRIOS
# ==========================================
IBGE_CAMPINAS = 350950
PASTA_BRONZE = 'dados_sinan_parquet'
PASTA_SILVER = 'dados_processados_cnes_sinan_parquet'
CAMINHO_CNES_CSV = 'cnes_estabelecimentos.csv'

os.makedirs(PASTA_SILVER, exist_ok=True)

# Definição das colunas originais do ficheiro CNES
COL_MUNICIPIO = 'CO_IBGE'
COL_CNES = 'CO_CNES'
COL_NOME = 'NO_FANTASIA'
COL_LAT = 'NU_LATITUDE'
COL_LON = 'NU_LONGITUDE'

# ==========================================
# 2. CARREGAMENTO DAS PARTIÇÕES DO SINAN (BRONZE)
# ==========================================
print("A carregar as partições do SINAN (Camada Bronze)...")

if not os.path.exists(PASTA_BRONZE):
    raise FileNotFoundError(f"A pasta '{PASTA_BRONZE}' não foi encontrada. Rode o Notebook 01 primeiro.")

ficheiros_sinan = [os.path.join(PASTA_BRONZE, f) for f in os.listdir(PASTA_BRONZE) if f.endswith('.parquet')]
lista_dfs = []

for f in ficheiros_sinan:
    df_temp = pd.read_parquet(f)
    # Extrai o prefixo (doença) antes do primeiro underscore para manter a linhagem
    df_temp['NOME_DOENCA'] = os.path.basename(f).split('_')[0]
    lista_dfs.append(df_temp)

df_sinan = pd.concat(lista_dfs, ignore_index=True)
print(f"Total de {len(df_sinan)} notificações epidemiológicas prontas para cruzamento.")

# ==========================================
# 3. CARREGAMENTO E NORMALIZAÇÃO DO CNES
# ==========================================
print("\nA ler a base territorial estática do CNES (Ficheiro CSV)...")
df_cnes = pd.read_csv(
    CAMINHO_CNES_CSV,
    sep=';',
    encoding='latin1',
    dtype=str,
    quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

# Remove as aspas físicas que o QUOTE_NONE mantém nos nomes das colunas
df_cnes.columns = df_cnes.columns.str.replace('"', '', regex=False).str.strip()

# Remove as aspas físicas dos valores para não quebrar os futuros MERGES/JOINS
for col in df_cnes.columns:
    df_cnes[col] = df_cnes[col].str.replace('"', '', regex=False).str.strip()

print("Base do CNES carregada e aspas normalizadas com sucesso!")

# Filtra apenas as unidades de Campinas usando o mapeamento CO_IBGE
df_cnes_campinas = df_cnes[df_cnes[COL_MUNICIPIO].str.startswith('350950', na=False)].copy()
df_cnes_campinas = df_cnes_campinas[[COL_CNES, COL_NOME, COL_LAT, COL_LON]]

# Renomeia temporariamente as colunas geográficas para evitar colisão de nomes com o SINAN
df_cnes_campinas.rename(columns={
    COL_CNES: 'ID_UNIDADE',
    COL_LAT: 'LAT_CNES',
    COL_LON: 'LON_CNES'
}, inplace=True)

print(f"Total de {len(df_cnes_campinas)} estabelecimentos de saúde mapeados em Campinas.")

# ==========================================
# 4. HIGIENIZAÇÃO DE CHAVES E CRUZAMENTO
# ==========================================
print("\nA executar a higienização de chaves e o cruzamento espacial...")

# Limpeza rigorosa das chaves de ligação (remoção de .0 e preenchimento de 7 dígitos)
df_sinan['ID_UNIDADE'] = df_sinan['ID_UNIDADE'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', pd.NA).str.zfill(7)
df_cnes_campinas['ID_UNIDADE'] = df_cnes_campinas['ID_UNIDADE'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(7)

# Execução do Left Join
df_siest_geo = pd.merge(df_sinan, df_cnes_campinas, on='ID_UNIDADE', how='left')

# ==========================================
# 5. ESTEIRA DE IMPUTAÇÃO ESPACIAL OTIMIZADA
# ==========================================
print("🌍 A aplicar regras de negócio espaciais (CNES -> Município)...")
tqdm.pandas(desc="Geolocalizando")

LAT_CAMPINAS, LON_CAMPINAS = -22.9099, -47.0626

def enriquecer_coordenadas(row):
    # NÍVEL 1: Coordenada exata extraída do estabelecimento CNES
    lat_cnes = str(row.get('LAT_CNES', '')).strip().lower()
    if lat_cnes not in ["", "nan", "none", "null", "<na>", "0.0", "0"]:
        return row['LAT_CNES'], row['LON_CNES'], "EXATA_CNES"

    # NÍVEL 2: Centroide do Município (Tratamento para a restrição de sigilo de dados/LGPD)
    return LAT_CAMPINAS, LON_CAMPINAS, "CENTROIDE_MUNICIPIO"

resultados = df_siest_geo.progress_apply(enriquecer_coordenadas, axis=1, result_type="expand")

# ==========================================
# 6. AJUSTE DE TIPOS E GRAVAÇÃO DA CAMADA SILVER
# ==========================================
# Força a conversão para float puro para não gerar incompatibilidade binária no PyArrow/NoSQL
df_siest_geo['LATITUDE'] = pd.to_numeric(resultados[0], errors='coerce')
df_siest_geo['LONGITUDE'] = pd.to_numeric(resultados[1], errors='coerce')
df_siest_geo['PRECISAO_GEO'] = resultados[2]

# Remove as colunas auxiliares de junção, mantendo 'NO_FANTASIA' como metadado rico do hospital
df_siest_geo.drop(columns=['LAT_CNES', 'LON_CNES'], errors='ignore', inplace=True)

print("\n✅ Imputação Concluída! Resumo da Qualidade (Linhagem Geoespacial):")
print(df_siest_geo['PRECISAO_GEO'].value_counts())

# Salva o arquivo final estruturado
ficheiro_final = f'{PASTA_SILVER}/casos_geolocalizados.parquet'
df_siest_geo.to_parquet(ficheiro_final, index=False, compression='snappy')

print(f"\n🚀 Pipeline concluído com sucesso! Base única estruturada em: '{ficheiro_final}'")

A carregar as partições do SINAN (Camada Bronze)...
Total de 406740 notificações epidemiológicas prontas para cruzamento.

A ler a base territorial estática do CNES (Ficheiro CSV)...
Base do CNES carregada e aspas normalizadas com sucesso!
Total de 4867 estabelecimentos de saúde mapeados em Campinas.

A executar a higienização de chaves e o cruzamento espacial...
🌍 A aplicar regras de negócio espaciais (CNES -> Município)...


Geolocalizando: 100%|██████████| 406740/406740 [00:14<00:00, 28132.14it/s] 



✅ Imputação Concluída! Resumo da Qualidade (Linhagem Geoespacial):
PRECISAO_GEO
EXATA_CNES             393712
CENTROIDE_MUNICIPIO     13028
Name: count, dtype: int64

🚀 Pipeline concluído com sucesso! Base única estruturada em: 'dados_processados_cnes_sinan_parquet/casos_geolocalizados.parquet'
